In [12]:
#read in gene vs disease data
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm import LightFM

tsv_file = open("curated_gene_disease_associations.tsv")
df = pd.read_csv(tsv_file, delimiter="\t")

In [3]:
df

,geneId,geneSymbol,DSI,DPI,diseaseId,diseaseName,diseaseType,diseaseClass,diseaseSemanticType,score,EI,YearInitial,YearFinal,NofPmids,NofSnps,source
0,1,A1BG,0.700,0.538,C0019209,Hepatomegaly,phenotype,C23;C06,Finding,0.30,1.000,2017.0,2017.0,1,0,CTD_human
1,1,A1BG,0.700,0.538,C0036341,Schizophrenia,disease,F03,Mental or Behavioral Dysfunction,0.30,1.000,2015.0,2015.0,1,0,CTD_human
2,2,A2M,0.529,0.769,C0002395,Alzheimer's Disease,disease,C10;F03,Disease or Syndrome,0.50,0.769,1998.0,2018.0,3,0,CTD_human
3,2,A2M,0.529,0.769,C0007102,Malignant tumor of colon,disease,C06;C04,Neoplastic Process,0.31,1.000,2004.0,2019.0,1,0,CTD_human
4,2,A2M,0.529,0.769,C0009375,Colonic Neoplasms,group,C06;C04,Neoplastic Process,0.30,1.000,2004.0,2004.0,1,0,CTD_human
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84033,109580095,HBB-LCR,0.743,0.115,C0002875,Cooley's anemia,disease,C16;C15,Disease or Syndrome,0.30,NaN,NaN,NaN,0,0,CTD_human
84034,109580095,HBB-LCR,0.743,0.115,C0005283,beta Thalassemia,disease,C16;C15,Disease or Syndrome,0.30,NaN,NaN,NaN,0,0,CTD_human
84035,109580095,HBB-LCR,0.743,0.115,C0019025,Hemoglobin F Disease,disease,C16;C15,Disease or Syndrome,0.30,NaN,NaN,NaN,0,0,CTD_human
84036,109580095,HBB-LCR,0.743,0.115,C0085578,Thalassemia Minor,disease,C16;C15,Disease or Syndrome,0.30,NaN,NaN,NaN,0,0,CTD_human


In [4]:
#creating df with gene, disease, and association score
gd_df = df[['geneId', 'diseaseId', 'score']]

In [5]:
#finding unique genes and diseases
geneId_values = np.unique(gd_df['geneId'].values)
diseaseId_values = np.unique(gd_df['diseaseId'].values)

In [6]:
#Creating sparse matrix from csv formatted data
gd_matrix = pd.DataFrame(index = diseaseId_values, columns = geneId_values)

for disease in diseaseId_values:
    disease_df = gd_df[gd_df['diseaseId']==disease].set_index('geneId')
    gd_matrix.loc[disease] = disease_df['score']
    
gd_np = gd_matrix.to_numpy()
gd_np = np.array(gd_np, dtype=float)
gd_np[np.isnan(gd_np)] = 0
gd_coo = sp.sparse.coo_matrix(gd_np)

In [9]:
#splitting data into train and test and storing in data with geneIds
(train, test) = cross_validation.random_train_test_split(gd_coo, test_percentage=0.2, random_state=None)
data = {'train': train, 'test': test, 'gene_ids' : geneId_values}

In [13]:
#creating model
model = LightFM(loss='warp')

In [17]:
#training model
model.fit(data['train'], epochs=100, num_threads=2)

In [18]:
#measuring performance
test_precision = precision_at_k(model, data['test'], k=10).mean()
print(test_precision)
test_recall = recall_at_k(model, data['test'], k=10).mean()
print(test_recall)

0.02051572
0.06902037905051701


In [19]:
#function to recommend genes for a given disease. Takes in the model, data, and a list of
#diseases for genes to be recommend for. For each disease, prints out known associated genes
#and a list of recommended genes. Function modeled after tutorial 3, 'Recommendation Systems
#- Learn Python for Data Science' on LightFM github. 

def sample_recommendation(model, data, diseases):
    
    #number of diseases and genes in training dataset
    n_diseases, n_genes = data['train'].shape
    
    #generate gene recommendations for each disease
    for disease in diseases:
        disease_id = df[df['diseaseName'] == disease]['diseaseId'].tolist()[0]
        disease_id_num = list(diseaseId_values).index(disease_id)
        #genes known to relate to the disease
        known_positives = data['gene_ids'][data['train'].tocsr()[disease_id_num].indices]
#         print(known_positives)
        
        #genes model predicts that relate to disease
        scores = model.predict(disease_id_num, np.arange(n_genes))
#         print(np.argsort(-scores))
#         print(max(np.argsort(-scores)))
        #rank from most related to least
        top_genes = data['gene_ids'][np.argsort(-scores)]
        
        #print results
        print('Disease: ' + disease + ' (' + str(disease_id_num) +')')
        print('   Known Positives:')
        
        for gene in known_positives[:]:
            x = df[df['geneId'] == gene]['geneSymbol'].tolist()[0]
            print('      %s' % x)
        
        print()
        print('   Recommended:')
        for gene in top_genes[:5]:
            x = df[df['geneId'] == gene]['geneSymbol'].tolist()[0]
            print('      %s' % x)
            
sample_recommendation(model, data, ['Cooley\'s anemia'])

Disease: Cooley's anemia (74)
   Known Positives:
      CAD
      DHODH
      HBB
      LCN2
      TFR2
      TFRC
      UMPS
      CACNA1H
      HAMP

   Recommended:
      HBB
      IDH2
      IGF1R
      CAD
      WWOX


In [20]:
#Sanity checks below for above function
print(data['gene_ids'][data['train'].tocsr()[3].indices])

[ 387 2052]


In [21]:
mask1 = gd_matrix[1]>0
mask2 = gd_matrix[196]>0
mask3 = gd_matrix[213]>0
# mask4 = gd_matrix[387]>0
print(gd_matrix[mask1 & mask2 & mask3])
print(df[df['diseaseId'] == 'C0019209'])

         1         2         9         10        12        13         \
C0019209       0.3       NaN       NaN       NaN       NaN       NaN   

         15        16        18        19         ... 105259599 105416157  \
C0019209       NaN       NaN       NaN       NaN  ...       NaN       NaN   

         106479631 106479784 106480868 106481323 106783499 107075310  \
C0019209       NaN       NaN       NaN       NaN       NaN       NaN   

         107305681 109580095  
C0019209       NaN       NaN  

[1 rows x 9703 columns]
       geneId geneSymbol    DSI    DPI diseaseId   diseaseName diseaseType  \
0           1       A1BG  0.700  0.538  C0019209  Hepatomegaly   phenotype   
1900      196        AHR  0.410  0.923  C0019209  Hepatomegaly   phenotype   
2273      213        ALB  0.317  0.962  C0019209  Hepatomegaly   phenotype   
2451      219    ALDH1B1  0.691  0.462  C0019209  Hepatomegaly   phenotype   
7186      815     CAMK2A  0.612  0.538  C0019209  Hepatomegaly   phenotype   


In [22]:
print(diseaseId_values[749])
i = list(diseaseId_values).index('C0019209')
print(i)

C0019209
749


In [23]:
print(df[df['diseaseName'] == 'Hepatomegaly']['diseaseId'].tolist()[0])

C0019209


In [24]:
print(df[df['geneId'] == 7124]['geneSymbol'].tolist()[0])

TNF


In [25]:
data['gene_ids'][3784]

7124